# Algorithmic Trading Group Project Notebook

This notebook is a beginner-friendly end-to-end template for the course group project. It includes **both recommended paths**:

1. Deep Reinforcement Learning for portfolio trading
2. Large Language Model based signal generation

The notebook is designed to be practical for a 3-person team and readable for beginners. It shares one common data pipeline, common benchmarks, then branches into the DRL path and the LLM path.

## How to use this notebook

Read from top to bottom once before changing anything.

What this notebook covers:
- data download and cleaning
- feature engineering
- benchmark strategies required by the brief
- a DRL portfolio strategy
- an LLM portfolio strategy
- comparison of all strategies
- optional Alpaca paper-trading handoff

Suggested team split:
- Person 1: data pipeline and benchmarks
- Person 2: main strategy logic
- Person 3: backtesting, Alpaca, and report assembly

In [ ]:
%pip install -q yfinance scipy gymnasium stable-baselines3 openai alpaca-py

In [ ]:
import os, json, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 200)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    import gymnasium as gym
    from gymnasium import spaces
    GYM_AVAILABLE = True
except ImportError:
    GYM_AVAILABLE = False

try:
    from stable_baselines3 import PPO
    SB3_AVAILABLE = True
except ImportError:
    SB3_AVAILABLE = False

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

print({"GYM_AVAILABLE": GYM_AVAILABLE, "SB3_AVAILABLE": SB3_AVAILABLE, "OPENAI_AVAILABLE": OPENAI_AVAILABLE})

## Project configuration

A beginner mistake is changing too many things at once. We define the asset universe, dates, transaction cost assumptions, and rebalance schedule in one place so the experiment is reproducible.

In [ ]:
@dataclass
class ProjectConfig:
    tickers: List[str]
    benchmark_ticker: str
    start_date: str
    end_date: str
    train_end: str
    val_end: str
    rebalance_frequency: str
    risk_free_rate_annual: float
    transaction_cost_bps: float
    initial_capital: float

config = ProjectConfig(
    tickers=["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META"],
    benchmark_ticker="SPY",
    start_date="2018-01-01",
    end_date="2025-12-31",
    train_end="2022-12-31",
    val_end="2024-12-31",
    rebalance_frequency="W-FRI",
    risk_free_rate_annual=0.02,
    transaction_cost_bps=10.0,
    initial_capital=100000.0,
)
config

## Download data

We use daily data for a small set of large liquid US stocks plus `SPY` as the broad-market benchmark. This is a good beginner choice because it is easy to explain, cheaper to compute, and consistent with the project requirements.

In [ ]:
all_tickers = config.tickers + [config.benchmark_ticker]
raw_data = yf.download(all_tickers, start=config.start_date, end=config.end_date, auto_adjust=True, progress=False)
if raw_data.empty:
    raise ValueError("No data downloaded. Check internet access, tickers, or date range.")
close_prices = raw_data["Close"].copy().dropna(how="all").ffill().dropna()
volume_data = raw_data["Volume"].copy().dropna(how="all").reindex(close_prices.index).ffill()
asset_close = close_prices[config.tickers].copy()
benchmark_close = close_prices[config.benchmark_ticker].copy()
daily_returns = asset_close.pct_change().fillna(0.0)
benchmark_returns = benchmark_close.pct_change().fillna(0.0)
print(asset_close.shape)
asset_close.tail()

## Feature engineering

A model should not look only at raw prices. We create simple features that summarize recent market behavior:
- short-term return
- medium-term return
- rolling volatility
- moving-average distance
- RSI-style momentum
- abnormal volume

These features are beginner-friendly because they are intuitive and commonly used in trading research.

In [ ]:
def compute_rsi(price_series: pd.Series, window: int = 14) -> pd.Series:
    delta = price_series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).fillna(50)

def build_feature_panel(close_df: pd.DataFrame, volume_df: pd.DataFrame) -> pd.DataFrame:
    feature_frames = []
    for ticker in close_df.columns:
        price = close_df[ticker]
        volume = volume_df[ticker]
        df = pd.DataFrame(index=close_df.index)
        df["ticker"] = ticker
        df["close"] = price
        df["ret_1d"] = price.pct_change(1)
        df["ret_5d"] = price.pct_change(5)
        df["ret_20d"] = price.pct_change(20)
        df["vol_20d"] = price.pct_change().rolling(20).std()
        df["ma_ratio_10"] = price / price.rolling(10).mean() - 1.0
        df["ma_ratio_30"] = price / price.rolling(30).mean() - 1.0
        df["rsi_14"] = compute_rsi(price, 14)
        df["volume_z"] = (np.log1p(volume) - np.log1p(volume).rolling(20).mean()) / np.log1p(volume).rolling(20).std()
        feature_frames.append(df)
    features = pd.concat(feature_frames, axis=0)
    features.index.name = "date"
    features = features.reset_index().sort_values(["date", "ticker"]).reset_index(drop=True)
    return features.replace([np.inf, -np.inf], np.nan)

features_long = build_feature_panel(asset_close, volume_data[config.tickers]).dropna().reset_index(drop=True)
feature_columns = ["ret_1d", "ret_5d", "ret_20d", "vol_20d", "ma_ratio_10", "ma_ratio_30", "rsi_14", "volume_z"]
feature_pivot = features_long.pivot(index="date", columns="ticker", values=feature_columns).sort_index()
feature_pivot.columns = [f"{feature}_{ticker}" for feature, ticker in feature_pivot.columns]
feature_pivot = feature_pivot.dropna()
common_index = asset_close.index.intersection(feature_pivot.index)
asset_close = asset_close.loc[common_index]
daily_returns = daily_returns.loc[common_index]
benchmark_close = benchmark_close.loc[common_index]
benchmark_returns = benchmark_returns.loc[common_index]
feature_pivot = feature_pivot.loc[common_index]
train_dates = common_index[common_index <= pd.Timestamp(config.train_end)]
val_dates = common_index[(common_index > pd.Timestamp(config.train_end)) & (common_index <= pd.Timestamp(config.val_end))]
test_dates = common_index[common_index > pd.Timestamp(config.val_end)]
print(len(train_dates), len(val_dates), len(test_dates))
feature_pivot.head()

## Shared backtesting utilities and required benchmarks

The brief requires benchmark comparison, including a broad market benchmark and a classic portfolio method such as mean-variance optimization. We therefore build three shared references:
- SPY buy-and-hold
- equal-weight portfolio
- mean-variance optimized portfolio

These benchmarks are important because a complex model should be judged against simple alternatives, not only by its own returns.

In [ ]:
TRADING_DAYS = 252

def normalize_weights(weights: np.ndarray, allow_short: bool = False) -> np.ndarray:
    weights = np.asarray(weights, dtype=float)
    if not allow_short:
        weights = np.clip(weights, 0, None)
    total = weights.sum()
    return np.ones_like(weights) / len(weights) if total <= 0 else weights / total

def compute_portfolio_returns(weight_history: pd.DataFrame, returns_df: pd.DataFrame, transaction_cost_bps: float = 0.0) -> pd.Series:
    weight_history = weight_history.reindex(returns_df.index).ffill().fillna(0.0)
    gross = (weight_history.shift(1).bfill() * returns_df).sum(axis=1)
    turnover = weight_history.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (transaction_cost_bps / 10000.0)
    return gross - cost

def compute_metrics(return_series: pd.Series, risk_free_rate_annual: float = 0.0) -> pd.Series:
    return_series = return_series.dropna()
    wealth = (1 + return_series).cumprod()
    annual_return = wealth.iloc[-1] ** (TRADING_DAYS / len(return_series)) - 1
    annual_vol = return_series.std() * np.sqrt(TRADING_DAYS)
    daily_rf = (1 + risk_free_rate_annual) ** (1 / TRADING_DAYS) - 1
    excess = return_series - daily_rf
    sharpe = np.nan if excess.std() == 0 else excess.mean() / excess.std() * np.sqrt(TRADING_DAYS)
    drawdown = wealth / wealth.cummax() - 1
    return pd.Series({
        "Cumulative Return": wealth.iloc[-1] - 1,
        "Annualized Return": annual_return,
        "Annualized Volatility": annual_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": drawdown.min(),
    })

def wealth_curve(return_series: pd.Series, initial_capital: float = 1.0) -> pd.Series:
    return initial_capital * (1 + return_series.fillna(0.0)).cumprod()

def plot_wealth_curves(curves: Dict[str, pd.Series], title: str) -> None:
    plt.figure(figsize=(12, 6))
    for label, curve in curves.items():
        plt.plot(curve.index, curve.values, label=label, linewidth=2)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value")
    plt.legend()
    plt.show()

def build_equal_weight_history(dates: pd.Index, tickers: List[str]) -> pd.DataFrame:
    w = np.ones(len(tickers)) / len(tickers)
    return pd.DataFrame(index=dates, columns=tickers, data=np.tile(w, (len(dates), 1)))

def mean_variance_weights(expected_returns: np.ndarray, cov_matrix: np.ndarray, risk_aversion: float = 4.0) -> np.ndarray:
    n = len(expected_returns)
    def objective(w):
        return -(np.dot(w, expected_returns) - risk_aversion * np.dot(w, cov_matrix @ w))
    result = minimize(objective, x0=np.ones(n) / n, bounds=[(0.0, 1.0)] * n, constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}])
    return np.ones(n) / n if not result.success else normalize_weights(result.x)

def build_mean_variance_history(returns_df: pd.DataFrame, rebalance_frequency: str, lookback_days: int = 60) -> pd.DataFrame:
    rebalance_dates = returns_df.resample(rebalance_frequency).last().index
    history = pd.DataFrame(index=returns_df.index, columns=returns_df.columns, dtype=float)
    current = np.ones(len(returns_df.columns)) / len(returns_df.columns)
    for date in returns_df.index:
        if date in rebalance_dates:
            loc = returns_df.index.get_loc(date)
            if loc >= lookback_days:
                window = returns_df.iloc[loc - lookback_days:loc]
                current = mean_variance_weights(window.mean().values * TRADING_DAYS, window.cov().values * TRADING_DAYS)
        history.loc[date] = current
    return history.ffill().dropna()

equal_weight_history = build_equal_weight_history(common_index, config.tickers)
mvo_weight_history = build_mean_variance_history(daily_returns, config.rebalance_frequency)
equal_weight_returns = compute_portfolio_returns(equal_weight_history, daily_returns, config.transaction_cost_bps)
mvo_returns = compute_portfolio_returns(mvo_weight_history, daily_returns, config.transaction_cost_bps)
spy_returns = benchmark_returns.copy()
benchmark_metrics = pd.DataFrame({
    "SPY Buy & Hold": compute_metrics(spy_returns.loc[test_dates], config.risk_free_rate_annual),
    "Equal Weight": compute_metrics(equal_weight_returns.loc[test_dates], config.risk_free_rate_annual),
    "Mean-Variance": compute_metrics(mvo_returns.loc[test_dates], config.risk_free_rate_annual),
}).T
benchmark_metrics.style.format("{:.4f}")

In [ ]:
plot_wealth_curves({
    "SPY Buy & Hold": wealth_curve(spy_returns.loc[test_dates], config.initial_capital),
    "Equal Weight": wealth_curve(equal_weight_returns.loc[test_dates], config.initial_capital),
    "Mean-Variance": wealth_curve(mvo_returns.loc[test_dates], config.initial_capital),
}, "Benchmark Strategies on the Test Period")

# Path A: Deep Reinforcement Learning

Plain-English idea: the agent sees recent market features, chooses portfolio weights, then receives a reward based on realized next-period return after transaction costs.

For beginners, the key RL concepts are:
- state: the current market feature vector
- action: portfolio weights across assets
- reward: the next-period net portfolio return
- episode: one full walk through the dataset

In [ ]:
drl_train_features = feature_pivot.loc[train_dates]
drl_test_features = feature_pivot.loc[test_dates]
drl_train_returns = daily_returns.loc[train_dates]
drl_test_returns = daily_returns.loc[test_dates]

if GYM_AVAILABLE:
    class PortfolioTradingEnv(gym.Env):
        metadata = {"render_modes": []}
        def __init__(self, feature_df: pd.DataFrame, return_df: pd.DataFrame, transaction_cost_bps: float = 0.0):
            super().__init__()
            self.feature_df = feature_df.copy()
            self.return_df = return_df.copy()
            self.transaction_cost = transaction_cost_bps / 10000.0
            self.n_assets = return_df.shape[1]
            self.n_steps = len(feature_df)
            self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(feature_df.shape[1],), dtype=np.float32)
            self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.n_assets,), dtype=np.float32)
            self.reset()
        def _get_obs(self):
            return self.feature_df.iloc[self.current_step].values.astype(np.float32)
        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            self.current_step = 0
            self.prev_weights = np.ones(self.n_assets) / self.n_assets
            return self._get_obs(), {}
        def step(self, action):
            weights = normalize_weights(np.array(action), allow_short=False)
            asset_ret = self.return_df.iloc[self.current_step].values
            turnover = np.abs(weights - self.prev_weights).sum()
            reward = float(np.dot(weights, asset_ret) - turnover * self.transaction_cost)
            self.prev_weights = weights
            self.current_step += 1
            terminated = self.current_step >= self.n_steps
            obs = np.zeros(self.feature_df.shape[1], dtype=np.float32) if terminated else self._get_obs()
            return obs, reward, terminated, False, {"weights": weights}
else:
    print("Gymnasium is not available. Run the install cell first.")

In [ ]:
drl_model = None
if GYM_AVAILABLE and SB3_AVAILABLE:
    train_env = PortfolioTradingEnv(drl_train_features, drl_train_returns, config.transaction_cost_bps)
    drl_model = PPO("MlpPolicy", train_env, verbose=0, learning_rate=3e-4, n_steps=128, batch_size=64, gamma=0.99, seed=RANDOM_SEED)
    drl_model.learn(total_timesteps=10000)
    print("DRL training complete.")
else:
    print("DRL model was not trained because gymnasium or stable-baselines3 is missing.")

def generate_drl_weights(model, feature_df: pd.DataFrame, tickers: List[str]) -> pd.DataFrame:
    if model is None:
        eq = np.ones(len(tickers)) / len(tickers)
        return pd.DataFrame(index=feature_df.index, columns=tickers, data=np.tile(eq, (len(feature_df), 1)))
    rows = []
    for date, row in feature_df.iterrows():
        action, _ = model.predict(row.values.astype(np.float32), deterministic=True)
        rows.append(pd.Series(normalize_weights(action, allow_short=False), index=tickers, name=date))
    return pd.DataFrame(rows)

drl_weight_history = generate_drl_weights(drl_model, drl_test_features, config.tickers)
drl_test_returns_series = compute_portfolio_returns(drl_weight_history, drl_test_returns, config.transaction_cost_bps)
drl_metrics = compute_metrics(drl_test_returns_series, config.risk_free_rate_annual)
drl_metrics

# Path B: LLM-Based Trading Signals

Plain-English idea: the language model acts like a structured decision engine. We give it recent features for each asset and require JSON output with long-only portfolio weights.

Important beginner rule: do not let the model answer with free-form paragraphs if you want a reliable backtest. Always force a parseable output format.

In [ ]:
rebalance_dates = asset_close.resample(config.rebalance_frequency).last().index
rebalance_dates = [d for d in rebalance_dates if d in feature_pivot.index and d in daily_returns.index]

def build_snapshot_for_date(date: pd.Timestamp) -> pd.DataFrame:
    date_slice = features_long[features_long["date"] == date].set_index("ticker")
    rows = []
    for ticker in config.tickers:
        src = date_slice.loc[ticker]
        rows.append({
            "ticker": ticker,
            "close": float(src["close"]),
            "ret_5d": float(src["ret_5d"]),
            "ret_20d": float(src["ret_20d"]),
            "vol_20d": float(src["vol_20d"]),
            "ma_ratio_10": float(src["ma_ratio_10"]),
            "ma_ratio_30": float(src["ma_ratio_30"]),
            "rsi_14": float(src["rsi_14"]),
        })
    return pd.DataFrame(rows)

llm_snapshot_example = build_snapshot_for_date(rebalance_dates[-5])
llm_snapshot_example

In [ ]:
def build_llm_prompt(snapshot_df: pd.DataFrame) -> str:
    asset_lines = []
    for _, row in snapshot_df.iterrows():
        asset_lines.append(
            f"Ticker={row['ticker']}, close={row['close']:.2f}, ret_5d={row['ret_5d']:.4f}, ret_20d={row['ret_20d']:.4f}, vol_20d={row['vol_20d']:.4f}, ma_ratio_10={row['ma_ratio_10']:.4f}, ma_ratio_30={row['ma_ratio_30']:.4f}, rsi_14={row['rsi_14']:.2f}"
        )
    asset_block = "
".join(asset_lines)
    return f"""
You are a cautious portfolio allocation assistant.
Use only the information given. Produce long-only weights that sum to 1.0. Prefer diversified allocations and consider both momentum and risk.

Asset snapshot:
{asset_block}

Return valid JSON only with this exact schema:
{{
  "weights": {{"AAPL": 0.10, "MSFT": 0.15}},
  "summary": "One short sentence.",
  "risk_note": "One short sentence."
}}
""".strip()

print(build_llm_prompt(llm_snapshot_example)[:1000])

In [ ]:
LLM_MODE = os.getenv("LLM_MODE", "mock").lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

def mock_llm_portfolio(snapshot_df: pd.DataFrame) -> Dict:
    scores = []
    for _, row in snapshot_df.iterrows():
        momentum_score = 0.45 * row["ret_20d"] + 0.35 * row["ret_5d"]
        trend_score = 0.15 * row["ma_ratio_10"] + 0.15 * row["ma_ratio_30"]
        risk_penalty = 0.30 * row["vol_20d"]
        rsi_penalty = 0.02 * max(row["rsi_14"] - 70, 0)
        scores.append(max(momentum_score + trend_score - risk_penalty - rsi_penalty, 0))
    weights = normalize_weights(np.array(scores), allow_short=False)
    return {
        "weights": {ticker: float(weight) for ticker, weight in zip(snapshot_df["ticker"], weights)},
        "summary": "Mock LLM favored stronger momentum with a penalty for risk.",
        "risk_note": "Mock mode is for offline development and should not be presented as a true LLM experiment.",
    }

def call_openai_portfolio(snapshot_df: pd.DataFrame, model: str, api_key: str) -> Dict:
    if not OPENAI_AVAILABLE:
        raise ImportError("The openai package is not installed.")
    if not api_key:
        raise ValueError("OPENAI_API_KEY is missing.")
    client = OpenAI(api_key=api_key)
    response = client.responses.create(model=model, input=build_llm_prompt(snapshot_df), temperature=0)
    parsed = json.loads(response.output_text)
    parsed_weights = [parsed["weights"].get(ticker, 0.0) for ticker in config.tickers]
    parsed["weights"] = {ticker: float(w) for ticker, w in zip(config.tickers, normalize_weights(np.array(parsed_weights)))}
    return parsed

def get_llm_signal(snapshot_df: pd.DataFrame) -> Dict:
    return call_openai_portfolio(snapshot_df, OPENAI_MODEL, OPENAI_API_KEY) if LLM_MODE == "openai" else mock_llm_portfolio(snapshot_df)

example_llm_output = get_llm_signal(llm_snapshot_example)
example_llm_output

In [ ]:
def build_llm_weight_history(rebalance_dates: List[pd.Timestamp], returns_df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
    history = pd.DataFrame(index=returns_df.index, columns=config.tickers, dtype=float)
    logs = []
    current = np.ones(len(config.tickers)) / len(config.tickers)
    for date in returns_df.index:
        if date in rebalance_dates:
            snapshot = build_snapshot_for_date(date)
            signal = get_llm_signal(snapshot)
            raw = np.array([signal["weights"].get(ticker, 0.0) for ticker in config.tickers])
            current = normalize_weights(raw, allow_short=False)
            logs.append({
                "date": date,
                "weights": {ticker: float(weight) for ticker, weight in zip(config.tickers, current)},
                "summary": signal.get("summary", ""),
                "risk_note": signal.get("risk_note", ""),
            })
        history.loc[date] = current
    return history.ffill().dropna(), logs

test_rebalance_dates = [d for d in rebalance_dates if d in test_dates]
llm_weight_history, llm_logs = build_llm_weight_history(test_rebalance_dates, drl_test_returns)
llm_test_returns_series = compute_portfolio_returns(llm_weight_history, drl_test_returns, config.transaction_cost_bps)
llm_metrics = compute_metrics(llm_test_returns_series, config.risk_free_rate_annual)
llm_metrics

In [ ]:
comparison_table = pd.DataFrame({
    "SPY Buy & Hold": compute_metrics(spy_returns.loc[test_dates], config.risk_free_rate_annual),
    "Equal Weight": compute_metrics(equal_weight_returns.loc[test_dates], config.risk_free_rate_annual),
    "Mean-Variance": compute_metrics(mvo_returns.loc[test_dates], config.risk_free_rate_annual),
    "DRL PPO": drl_metrics,
    f"LLM ({LLM_MODE})": llm_metrics,
}).T
comparison_table.style.format("{:.4f}")

In [ ]:
plot_wealth_curves({
    "SPY Buy & Hold": wealth_curve(spy_returns.loc[test_dates], config.initial_capital),
    "Equal Weight": wealth_curve(equal_weight_returns.loc[test_dates], config.initial_capital),
    "Mean-Variance": wealth_curve(mvo_returns.loc[test_dates], config.initial_capital),
    "DRL PPO": wealth_curve(drl_test_returns_series, config.initial_capital),
    f"LLM ({LLM_MODE})": wealth_curve(llm_test_returns_series, config.initial_capital),
}, "All Strategy Wealth Curves on the Test Period")

## Optional Alpaca paper-trading handoff

The brief requires Alpaca paper trading. The helper below converts target portfolio weights into market order requests for a paper account.

Safety note: use Alpaca paper trading only. Do not point this workflow at a live account unless you explicitly intend to trade real money.

In [ ]:
ALPACA_READY = False
try:
    from alpaca.trading.client import TradingClient
    from alpaca.trading.requests import MarketOrderRequest
    from alpaca.trading.enums import OrderSide, TimeInForce
    ALPACA_READY = True
except ImportError:
    ALPACA_READY = False

def choose_target_weights(strategy_name: str) -> pd.Series:
    if strategy_name == "drl":
        return drl_weight_history.iloc[-1]
    if strategy_name == "llm":
        return llm_weight_history.iloc[-1]
    if strategy_name == "mvo":
        return mvo_weight_history.loc[test_dates].iloc[-1]
    return equal_weight_history.loc[test_dates].iloc[-1]

def build_paper_orders(target_weights: pd.Series, latest_prices: pd.Series, notional_capital: float = 10000.0):
    if not ALPACA_READY:
        raise ImportError("alpaca-py is not available. Run the install cell first.")
    api_key = os.getenv("ALPACA_API_KEY")
    secret_key = os.getenv("ALPACA_SECRET_KEY")
    paper = TradingClient(api_key=api_key, secret_key=secret_key, paper=True)
    orders = []
    for ticker, weight in target_weights.items():
        dollar_amount = float(weight) * notional_capital
        qty = max(int(dollar_amount / latest_prices[ticker]), 0)
        if qty == 0:
            continue
        orders.append(MarketOrderRequest(symbol=ticker, qty=qty, side=OrderSide.BUY, time_in_force=TimeInForce.DAY))
    return orders, paper

## Final beginner checklist

Before submission, check the following:
- multi-asset portfolio used
- mean-variance benchmark included
- SPY or another broad-market benchmark included
- real out-of-sample test period used
- transaction costs included
- Alpaca paper-trading workflow prepared
- notebook readable enough for a beginner teammate to rerun and explain